In [ ]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:90% !important;}
div.cell.code_cell.rendered{width:100%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:12pt;}
div.output {font-size:12pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:12pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:12px;}
table {margin-left:0 !important; margin-right: auto;}
</style>
"""))

<font size="5" color="black">ch8. OpenAI function calling</font>


# OpenAI API Function Calling


# 1. 함수 호출 기능

* 사전에 정의된 외부 함수에 대한 필요시 실행 지원 도구
* 자체 지식 미보유 영역(실시간 정보 조회, 복잡한 계산 작업 등)의 외부 함수 위임 처리
* 날씨 문의 시 날씨 API 연동 함수 호출을 통한 최신 기온 수집 및 응답 구현 사례
* 실시간/외부 데이터(날씨, 주가, 뉴스 등)의 수신 및 답변 반영을 통한 정보 최신화
* 모델 한계 보완(수학 계산, 데이터베이스 질의 등 직접 처리 불가 영역의 외부 로직 해결)
* 개발자 정의 임의 함수를 활용한 모델 기능 확장 및 특정 도메인/서비스 특화 어시스턴트 구현
* 대부분의 최신 OpenAI 모델(gpt-4.1 계열, gpt-4o 계열 등) 내 함수 호출 기능 지원
* 질문 의도 기반의 적절한 함수 선택 및 필요한 인수(Arguments) 포함 호출 형식 응답 반환 메커니즘

# 2. API 키 설정 및 OpenAI 클라이언트 초기화

In [ ]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
client = OpenAI()

# 3. 외부 함수 정의 및 테스트

**날씨 정보 수집 함수(`get_weather`) 정의 및 실습 목적**
* 함수 호출(Function Calling) 기능 체험을 위한 날씨 정보 조회 예시 함수 구현
* "서울 날씨 어때요?" 질문 수신 시 어시스턴트의 함수 호출 및 실시간 날씨 정보 기반 답변 프로세스


**`get_weather` 함수의 상세 구현 메커니즘**
* 위도(`latitude`) 및 경도(`longitude`) 좌표 인자 수신 구조
* 오픈 메테오(Open-Meteo) 공개 기상 API를 활용한 특정 좌표의 현재 기온(섭씨) 데이터 수집
* `requests` 라이브러리 기반 API 호출 및 응답 JSON 내 온도 값 추출·반환 로직


**예시 좌표 기반 함수 동작 검증**
* 함수 정의 후 예시 좌표 입력을 통한 정상 동작 여부 확인
* 서울 대략 좌표(위도 `37.484642`, 경도 `126.930063`) 입력 시 현재 온도 값 정상 반환 확인 및 출력 테스트

In [ ]:
import requests
import json

def get_weather(latitude=37.484859, longitude=126.930086):
    url = f"https://api.open-meteo.com/v1/forecast?latitude={latitude}&longitude={longitude}&current=temperature_2m"
    response = requests.get(url)
    data = response.json()
    current = data.get('current')
    if current is not None:
        return current.get("temperature_2m")
    else:
        return None
    
get_weather()

# 4. 함수 툴 설정 및 모델 응답 생성


**OpenAI API 대상 정의 함수 툴(`tool`) 등록 및 모델 호출 유도**
* 모델의 특정 함수 호출 유도를 위한 API 내 함수 툴 등록 절차
* `tools` 리스트 기반의 함수 정보 구성 프로세스


**`tools` 리스트 내 함수 정보 스키마 정의 규격**
* 함수 툴별 필수 구성 요소: 이름(`name`), 설명(`description`), 매개변수(`parameters`) 스키마
* `type: "function"` 설정을 통한 툴 유형 정의
* `name: "get_weather"` 식별자 지정
* `description: "저장된 좌표의 현재 온도를 섭씨 단위로 구합니다"` 목적 설명 작성
* `parameters` 필드 내 요구 인자(숫자형 `latitude`, `longitude` 및 둘 다 필수)의 JSON 스키마 명시


**대화 메시지(`messages`) 구성 및 API 호출**
* 사용자의 질문 메시지("오늘 서울 날씨 어때요?") 준비 및 `messages` 리스트 추가
* 사용자 메시지 및 정의된 `tools` 파라미터를 포함한 `client.chat.completions.create` 호출 실행
* `tools` 파라미터 전달을 통한 모델의 답변 과정 내 함수 호출 권한 부여


**모델의 함수 호출 결정 및 응답 반환 메커니즘**
* 함수 호출 결정 시 일반 텍스트 응답 대신 **함수 호출 요청** 포맷 반환
* 모델의 직접적인 답변 제공 대신 특정 인수(Arguments) 기반 `get_weather` 함수 호출 정보 반환 동작
* ChatCompletion API 호출 실행 및 `completion` 변수 대상 결과 저장 프로세스

In [ ]:
messages = [{'role':'user','content':'오늘 서울 날씨 어때요?'}]
tools = [{
    'type':'function',
    'function':{
        'name':'get_weather',
        'description':'지정된 좌표(위도, 경도)의 현재 온도를 섭씨 단위로 반환합니다',
        'parameters':{
            'type':'object',
            'properties':{
                'latitude':{'type':'number'},
                'longitude':{'type':'number'}
            }, # properties
            'required':['latitude','longitude'], # 필수 파라미터 지정
            'additionalProperties':False # 정의된 파라미터 외 추가 파라미터 금지
        }, #parameters
        'strict':True # 모델이 정의된 스키마를 엄격히 따르도록 강제
    } # function
}] # tools

completion = client.chat.completions.create(
    model='gpt-4o-mini',
    messages=messages,
    tools=tools,
    parallel_tool_calls=False # 하나의 함수만 호출 ; 기본값 : True (병렬 연산)
)

completion

In [ ]:
messages = [{'role':'user','content':'오늘 제주 날씨 어때요?'}]
tools = [{
    'type':'function',
    'function':{
        'name':'get_weather',
        'description':'지정된 좌표(위도, 경도)의 현재 온도를 섭씨 단위로 반환합니다',
        'parameters':{
            'type':'object',
            'properties':{
                'latitude':{'type':'number'},
                'longitude':{'type':'number'}
            }, # properties
            'required':['latitude','longitude'], # 필수 파라미터 지정
            'additionalProperties':False # 정의된 파라미터 외 추가 파라미터 금지
        }, #parameters
        'strict':True # 모델이 정의된 스키마를 엄격히 따르도록 강제
    } # function
}] # tools

completion = client.chat.completions.create(
    model='gpt-4o-mini',
    messages=messages,
    tools=tools,
    parallel_tool_calls=False # 하나의 함수만 호출 ; 기본값 : True (병렬 연산)
)

completion

# 5. 함수 호출 결과 처리 및 최종 답변

**함수 호출 결과 처리 및 최종 답변 전달 프로세스**
* 모델의 함수 호출 요청 반환 시 개발자 측의 해당 함수 실제 실행 요구
* 실행된 함수 결과의 어시스턴트 대상 재전달 필요

**모델 요청 함수 호출 정보 확인 기법**
* `completion.choices[0].message.tool_calls` 속성을 통한 함수 호출 요청 정보의 확인
* 요청 정보 내 호출 대상 함수 이름 및 전달 인자(Arguments) 정보 포함

In [ ]:
completion.choices[0].message.tool_calls

In [ ]:
for tool_call in completion.choices[0].message.tool_calls:
    name = tool_call.function.name
    arg = tool_call.function.arguments
    arg = json.loads(arg)
    latitude = arg.get('latitude')
    longitude = arg.get('longitude')
    if name == 'get_weather':
        result = get_weather(latitude, longitude)
        print(result)

함수 실행 결과를 어시스턴트 대화에 추가해 보겠습니다. `messages.append(...)`를 사용하여, 먼저 어시스턴트의 함수 호출 메시지를 대화 내역에 넣고, 이어서 'tool' 역할의 메시지를 추가합니다. 이 'tool' 메시지에는 함수 호출의 `id`와 실행 결과를 문자열로 담았습니다. 이렇게 하면 OpenAI API는 해당 함수 호출에 대한 결과를 받았다고 인식하게 됩니다.

In [ ]:
# model의 assistant message 추가
print('추가 전(user message만) :',messages)
messages.append(completion.choices[0].message)

In [ ]:
# tool message 추가
messages.append({'role':'tool','tool_call_id':tool_call.id, 'content':str(result)})
print('추가 후 :', messages)

In [ ]:
from pprint import pprint
pprint(messages)

**대화 내역 기반 최종 답변 요청 및 모델 응답 생성**
* 누적 대화 내역(`messages`)을 바탕으로 한 모델 대상 최종 답변 재요청 프로세스
* `messages` 내 사용자 질문, 함수 호출 요청, 함수 실행 결과의 포괄적 포함
* 포함된 맥락 정보를 활용한 사용자 맞춤형 최종 답변 생성 동작


**함수 호출 메시지 순서 및 짝 맞춤 규칙 (주의사항)**
* 모델의 함수 호출 응답 바로 직후 대응하는 `'tool'` 유형 메시지 배치 필수성
* 다중 함수 동시 호출 시 일부 결과 메시지 누락에 따른 "처리되지 않은 tool_call_id 존재" 오류 발생 메커니즘

```
우리 예시는 tools 호출 시 parallel_tool_calls=False로 지정해 두었기 때문에, 모델이 함수를 한 번에 하나씩만 호출합니다. 그래서 tool_call 하나에 결과 메시지 하나만 짝지어 넣어도 누락이 생기지 않고, 아래 요청은 정상적으로 최종 답변을 받아옵니다. (만약 parallel_tool_calls를 켜서 여러 함수가 동시에 호출되는 경우라면, 모든 tool_call에 대해 각각 결과 메시지를 추가해야 오류 없이 진행됩니다.)
```

In [ ]:
response = client.chat.completions.create(
    model='gpt-4o-mini',
    messages=messages,
    tools=tools
)

In [ ]:
print(response.choices[0].message.content)